In [8]:
pip install alerce

Note: you may need to restart the kernel to use updated packages.


In [1]:
from joblib import load
import numpy as np
import pandas as pd
import os 
from alerce.core import Alerce
from astropy.time import Time
import pandas as pd

client = Alerce()


In [3]:
artifacts = load("iforest_artifacts.joblib")

iso = artifacts["iso"]
knn_imp = artifacts["knn_imp"]
numeric_feature_cols = artifacts["numeric_feature_cols"]

In [8]:
objs = client.query_objects(
    survey="lsst",
    page_size=5,
    format="pandas",
)
objs.head()

,oid,tid,sid,meanra,meandec,sigmara,sigmadec,firstmjd,lastmjd,deltamjd,n_det,n_forced,n_non_det,stellar,class_name,classifier_name,classfier_version,probability,ranking
0,313998569187573806,1,1,149.131364,1.067110,0.000016,0.000015,61057.300498,61057.300498,0.000000,1,0,0,None,asteroid,stamp_classifier_rubin,None,0.997234,1
1,313994143567708179,1,1,51.534186,-27.024135,0.000013,0.000013,61056.203884,61056.203884,0.000000,1,0,0,None,asteroid,stamp_classifier_rubin,None,0.997229,1
2,170050515691372720,1,1,187.445344,8.009743,0.000003,0.000003,61095.218546,61096.339909,1.121363,2,1,0,None,SN,stamp_classifier_rubin,None,0.997226,1
3,313994143505842402,1,1,54.015369,-28.746231,0.000015,0.000016,61056.203884,61056.203884,0.000000,1,0,0,None,asteroid,stamp_classifier_rubin,None,0.997224,1
4,170028535477960720,1,1,185.987703,6.922455,0.000016,0.000028,61090.325355,61090.325355,0.000000,1,0,0,None,asteroid,stamp_classifier_rubin,None,0.997219,1


In [10]:
oid = int(objs["oid"].iloc[0])
oid

313998569187573806

In [12]:
detections = client.query_detections(
    survey = "lsst",
    oid = oid,
    format = "pandas",
)

print(detections.shape)
print(detections.columns)
detections.head()

(1, 102)
Index(['band', 'mjd', 'survey_id', 'ra', 'dec', 'oid', 'measurement_id',
       'parentDiaSourceId', 'diaObjectId', 'psfFlux',
       ...
       'pixelFlags_suspect', 'pixelFlags_suspectCenter', 'pixelFlags_streak',
       'pixelFlags_streakCenter', 'pixelFlags_injected',
       'pixelFlags_injectedCenter', 'pixelFlags_injected_template',
       'pixelFlags_injected_templateCenter', 'glint_trail', 'band_name'],
      dtype='object', length=102)


,band,mjd,survey_id,ra,dec,oid,measurement_id,parentDiaSourceId,diaObjectId,psfFlux,...,pixelFlags_suspect,pixelFlags_suspectCenter,pixelFlags_streak,pixelFlags_streakCenter,pixelFlags_injected,pixelFlags_injectedCenter,pixelFlags_injected_template,pixelFlags_injected_templateCenter,glint_trail,band_name
0,6,61057.300498,lsst,149.131364,1.06711,313998569187573806,313998569187573806,0,313998569187573806,13665.267,...,False,False,False,False,False,False,False,False,False,u


In [14]:
[c for c in detections.columns if "psfFlux" in c]

['psfFlux',
 'psfFluxErr',
 'psfFlux_flag',
 'psfFlux_flag_edge',
 'psfFlux_flag_noGoodPixels']

In [18]:
import numpy as np
import pandas as pd

def nJy_to_abmag(flux_njy):
    f_jy = np.asarray(flux_njy, dtype=float) * 1e-9
    with np.errstate(divide="ignore", invalid="ignore"):
        return -2.5 * np.log10(f_jy) + 8.90

def nJyerr_to_magerr(flux_njy, fluxerr_njy):
    flux = np.asarray(flux_njy, dtype=float)
    ferr = np.asarray(fluxerr_njy, dtype=float)
    out = np.full_like(flux, np.nan, dtype=float)
    mask = flux > 0
    out[mask] = (2.5 / np.log(10)) * (ferr[mask] / flux[mask])
    return out

def lsst_det_to_phot(det: pd.DataFrame) -> pd.DataFrame:
    df = det.copy()

    # band as string, prefer band_name (u,g,r,i,z,y)
    if "band_name" in df.columns:
        df["band"] = df["band_name"].astype(str)
    else:
        df["band"] = df["band"].astype(str)

    # optional quality cuts (safe defaults)
    # keep only rows where psfFlux is valid and not flagged badly
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["mjd", "psfFlux", "psfFluxErr", "band"])

    # compute mags
    df["mag"] = nJy_to_abmag(df["psfFlux"])
    df["magerr"] = nJyerr_to_magerr(df["psfFlux"], df["psfFluxErr"])

    phot = df[["mjd", "band", "mag", "magerr"]].copy()
    phot = phot.dropna(subset=["mag", "magerr"]).sort_values("mjd").reset_index(drop=True)
    return phot

phot = lsst_det_to_phot(detections)
phot

,mjd,band,mag,magerr
0,61057.300498,u,21.060955,0.023502


In [24]:
objs_rich = objs.sort_values("n_det", ascending=False).reset_index(drop=True)
objs_rich[["oid","n_det","n_forced","n_non_det","firstmjd","lastmjd","class_name","probability"]].head(20)

oid = int(objs_rich["oid"].iloc[0])
detections = client.query_detections(survey="lsst", oid=oid, format="pandas")
phot = lsst_det_to_phot(detections)
print("phot rows:", len(phot), "bands:", phot["band"].unique())
phot.head()

feats = compute_relaiss_lc_features(phot)   # returns dict or 1-row df

phot rows: 1 bands: ['r']


NameError: name 'compute_relaiss_lc_features' is not defined